In [1]:
import sklearn     # imports scikit-learn, a popular machine learning library in Python.
# scikit-learn (sklearn) is used for: Machine learning algorithms, Data preprocessing & Model evaluation
# It is built on NumPy, SciPy, and Matplotlib.

In [2]:
import sklearn      # Loads the scikit-learn machine learning library. Required before using any ML models, preprocessing, 
# or metrics.
import numpy as np  # NumPy is a core numerical library, Used for: Arrays, Random numbers, Linear algebra. np->alias
import pandas as pd # data manipulation library, used for: DataFrames, CSV loading, Data cleaning. pd->alias
# %matplotlib inline  # Jupyter magic command (Tells Matplotlib to display plots inside the notebook instead of opening 
# a separate window. ie, Required for inline visualization (If run in a .py file → error))
import matplotlib as mpl # <Import Matplotlib core> 
# Imports Matplotlib’s configuration system. Used to control: Fonts, Sizes, Styling defaults
import matplotlib.pyplot as plt # <Import plotting interface> 
# pyplot is the plotting API, Used for: Line plots, Histograms, Scatter plots. plt->alias
import warnings # Built-in Python module. Used to Control warning messages & Suppress unnecessary output

np.random.seed(42) # Set random seed
# Fixes the random number generator, Makes results reproducible, 
# Same random numbers every time you run the code
# Any integer works

# Configure Matplotlib axis label size
mpl.rc('axes', labelsize=14) # Sets default font size for X-axis label & Y-axis label. Applies to all plots
# Configure tick label sizes
mpl.rc('xtick', labelsize=12) # Controls font size of X-axis tick values
mpl.rc('ytick', labelsize=12) # # Controls font size of Y-axis tick values. Improves readability

warnings.filterwarnings(action="ignore", message="^interna; gelsd") # Suppress specific warnings
# What this does - Ignores warnings that Start with "interna; gelsd". 
# These warnings often come from Linear algebra solvers (LAPACK), SciPy / NumPy internals
# Why suppress them? They are Not user-actionable, Noisy in notebooks & Keeps output clean

In [3]:
HOUSING_PATH = "/cxldata/datasets/project/housing/housing.csv" #  path of the dataset
housing = pd.read_csv(HOUSING_PATH) # Read the dataset
housing.head() # Display the first few rows of the dataset

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [ ]:
## Explore dataset -1

housing.info()

In [ ]:
## Explore dataset -2

housing.describe()

In [ ]:
## Explore dataset -3

housing.hist(bins=50, figsize=(20,15))
plt.show()

In [ ]:
## Explore dataset -4

housing["median_income"].hist(bins=50, figsize=(20,15))
plt.show()

In [ ]:
## Explore dataset -5

housing["median_income"].hist()
plt.show()

In [ ]:
## Explore dataset -6

housing["income_cat"] = pd.cut(housing["median_income"],
                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                               labels=[1, 2, 3, 4, 5])
housing["income_cat"].hist()

In [ ]:
## Split dataset to test_strata 20% and train_starta 80%

# What this code achieves
# 1. Created a stratified split based on income categories
# 2. Ensured train and test have similar distributions
# 3. Removed the temporary column after use

from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]
    
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

In [ ]:
## Visualize geographic distribution of data

housing = strat_train_set.copy()

import matplotlib.image as mpimg
california_img=mpimg.imread('/cxldata/datasets/project/housing/california.png')
ax = housing.plot(kind="scatter", x="longitude", y="latitude", figsize=(10,7),
                       s=housing['population']/100, label="Population",
                       c="median_house_value", cmap=plt.get_cmap("jet"),
                       colorbar=False, alpha=0.4,
                      )
plt.imshow(california_img, extent=[-124.55, -113.80, 32.45, 42.05], alpha=0.5,
           cmap=plt.get_cmap("jet"))
plt.ylabel("Latitude", fontsize=14)
plt.xlabel("Longitude", fontsize=14)

prices = housing["median_house_value"]
tick_values = np.linspace(prices.min(), prices.max(), 11)
cbar = plt.colorbar(ticks=tick_values/prices.max())
cbar.ax.set_yticklabels(["$%dk"%(round(v/1000)) for v in tick_values], fontsize=14)
cbar.set_label('Median House Value', fontsize=16)

plt.legend(fontsize=16)
plt.show()

In [ ]:
# create new features from existing features

housing["rooms_per_household"] = housing["total_rooms"]/housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"]/housing["total_rooms"]
housing["population_per_household"]=housing["population"]/housing["households"]

In [ ]:
## Create correlation matrix - contains correlation values between all features

# Automatically ignore non-numeric columns
corr_matrix = housing.corr()
corr_matrix

In [ ]:
corr_matrix["median_house_value"].sort_values(ascending=False)

from pandas.plotting import scatter_matrix

attributes = ["median_house_value", "median_income", "total_rooms",
              "housing_median_age"]
scatter_matrix(housing[attributes], figsize=(12, 8))

In [ ]:
housing.describe()

In [ ]:
## Fill missing data with median

housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
housing_num = housing.drop("ocean_proximity", axis=1)
imputer.fit(housing_num)
X = imputer.transform(housing_num)
housing_tr = pd.DataFrame(X, columns=housing_num.columns,
                      index=housing.index)

In [ ]:
housing_cat = housing[["ocean_proximity"]]
housing_cat.head(10)

In [ ]:
## Handling Categorical attributes

from sklearn.preprocessing import OneHotEncoder
cat_encoder = OneHotEncoder()
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
housing_cat_1hot

housing_cat_1hot.toarray()

In [ ]:
## Creating custom transformer 
# It creates new, more meaningful features from existing ones.

from sklearn.base import BaseEstimator, TransformerMixin

rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room=True):
        self.add_bedrooms_per_room = add_bedrooms_per_room
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]  # More meaningful than raw total_rooms
        population_per_household = X[:, population_ix] / X[:, households_ix]  # Shows crowding
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix] # Indicates house quality
            return np.c_[X, rooms_per_household, population_per_household,
                         bedrooms_per_room] # Appends new columns to original data
        else:
            return np.c_[X, rooms_per_household, population_per_household]

attr_adder = CombinedAttributesAdder(add_bedrooms_per_room=False)
housing_extra_attribs = attr_adder.transform(housing.values)

# housing.values converts panda dataframe to numpy array

In [ ]:
## Creating transformation pipeline
# turning raw data into model-ready data in a clean, repeatable way
# Raw Data → Clean → Feature Engineering → Scaling → Encoding → Model-ready

col_names = "total_rooms", "total_bedrooms", "population", "households"
rooms_ix, bedrooms_ix, population_ix, households_ix = [
    housing.columns.get_loc(c) for c in col_names]

housing_extra_attribs = pd.DataFrame( # Converting back to DataFrame
    housing_extra_attribs,
    columns=list(housing.columns)+["rooms_per_household", "population_per_household"],
    index=housing.index)
housing_extra_attribs.head()

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median")),
        ('attribs_adder', CombinedAttributesAdder()),
        ('std_scaler', StandardScaler()), # Scales data (mean=0, std=1)
    # Many ML models perform better when features are scaled
    ])

housing_num_tr = num_pipeline.fit_transform(housing_num)

from sklearn.compose import ColumnTransformer

num_attribs = list(housing_num)
cat_attribs = ["ocean_proximity"]

full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", OneHotEncoder(), cat_attribs),
    ])


housing_prepared = full_pipeline.fit_transform(housing)

In [ ]:
# train Decision tree model

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(housing_prepared, housing_labels)

housing_predictions = tree_reg.predict(housing_prepared)

tree_mse = mean_squared_error(housing_labels, housing_predictions)
tree_rmse = np.sqrt(tree_mse)
tree_rmse

In [ ]:
# Train Random Forest Model

from sklearn.ensemble import RandomForestRegressor

# train the model
forest_reg = RandomForestRegressor(n_estimators=30, random_state=42)
forest_reg.fit(housing_prepared, housing_labels)

# predict using our model
housing_predictions = forest_reg.predict(housing_prepared)

# evaluate our model
forest_mse = mean_squared_error(housing_labels, housing_predictions)
forest_rmse = np.sqrt(forest_mse)
forest_rmse

In [ ]:
# Fine tune your model with Cross Validation

def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Standard deviation:", scores.std())
    
    
from sklearn.model_selection import cross_val_score

scores = cross_val_score(tree_reg, housing_prepared, housing_labels,
                         scoring="neg_mean_squared_error", cv=10)
tree_rmse_scores = np.sqrt(-scores)
display_scores(tree_rmse_scores)


In [ ]:
forest_scores = cross_val_score(forest_reg, housing_prepared, housing_labels,
                                scoring="neg_mean_squared_error", cv=10)
forest_rmse_scores = np.sqrt(-forest_scores)
display_scores(forest_rmse_scores)

In [ ]:
# Fine tune our model with Grid search

from sklearn.model_selection import GridSearchCV

param_grid = [
    {'n_estimators': [3, 10, 30], 'max_features': [2, 4, 6, 8]},
    {'bootstrap': [False], 'n_estimators': [3, 10], 'max_features': [2, 3, 4]},
  ]

In [ ]:
forest_reg = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
                           scoring='neg_mean_squared_error',
                           return_train_score=True)
grid_search.fit(housing_prepared, housing_labels)

In [ ]:
# best combination of parameters
grid_search.best_params_

In [ ]:
# best combination of estimators
grid_search.best_estimator_

In [ ]:
cvres = grid_search.cv_results_
for mean_score, params in zip(cvres["mean_test_score"], cvres["params"]):
    print(np.sqrt(-mean_score), params)

In [ ]:
final_model = grid_search.best_estimator_

X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

X_test_prepared = full_pipeline.transform(X_test)



In [ ]:
final_predictions = final_model.predict(X_test_prepared)

In [ ]:
final_mse = mean_squared_error(y_test, final_predictions)
final_rmse = np.sqrt(final_mse)